# Post5G - RAN Split

## Prerequisites

Before starting, make sure you:

- Have completed the [Simple Scenario](./10_simple.ipynb) and [Adding component](./11_add_components.ipynb) tutorials.
- Have a basic understanding of Python.
- Have a good understanding of the 5G architecture.
- Have a good understanding of the RAN split

## Experiment for this scenario

In this scenario we implement RAN splitting. We will consider the case of the
*CU - DU* split and the more advance case of *CU-CP/CU-UP - DU* split.

In our network on the one hand, the DU named `du1` is connected to the CU named
`cu`. On the other hand, the DU named `du2` is connected to the CU-CP named 
`cucp` and to the CU-UP named `cuup`. The UE `sim_ue1` is connected to `du1` and
the UE `sim_ue2` is connected to `du2` as illustrated in the figure below.

Each DU has its own Cell ID and PCI.

The 5G core is implemented with `Open5GS`, and the RAN is implemented with OAI.
The two UEs are implemented with `OAI` and the  radio channel is simulated with
rfsim.

![Split topology](figures/split.png)





## Metamodel and Deployement model definition

### Metamodel

The metamodel follows the same principle as in the other scenarios, except that
instead of defining a monolithic gNB, we define the split functions and how they
are connected.

We declare that the DU `du1` is connected to the CU `cu` by adding the `cu_cp`
and `cu_up` attributes to its `config` section. As `cu` implements at the same
time the CP and the UP, we reference it in both case.

On the contrary the DU `du2` is connected to the CU-CP `cucp` and to the CU-UP
`cuup`, we then explicitly reference them in the the `cu_cp` and `cu_up`
attributes of the DU. In addition, we have to specify that `cuup` has an E1
interface with the CU-CP `cucp`. This is why we have to add the `cu_cp`
attribute to the `config` section of `cuup`.

In addition, we specify a specific DU ID, CellID, and PCI to each DU for the
network to distinguish them.

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
ran:
    components:
      cu:
        type: cu

      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          gnbDuId: 1
          nrCellId: 1
          physicalCellId: 1
          cu_cp: cu
          cu_up: cu

      du2:
        type: du
        config:
          gnbDuId: 2
          nrCellId: 2
          physicalCellId: 2
          cu_cp: cucp
          cu_up: cuup
    
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du1
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du2
""")

### Deployment model

Beside declaring each component constituting the split RAN, there is no
specific settings to add to the deployment model to implement the split.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  implementation: oai
  components:
    cu:
      type: cu
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
    du1:
      type: du
    du2:
      type: du

ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized

nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue

provisioners:
  my-provisionner:
    type: bi
    params:
      infra: be-gent1-bi-vm1
      project: post5g-beta
      experiment: split_scenario
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
"""
)



## Provisioning the infrastructure

We can provision the infrastructure as in the [Simple Scenario tutorial](./10_simple.ipynb).

In [3]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-ues' provisioning] ***

PLAY [localhost] ********************************

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [4]:
from pathlib import Path
DIR="state/split"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core.

In [6]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-bsf' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-nssf' chart] *****

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-ausf' chart] *****

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-co

### Deploy the 5G RAN

We can now deploy the 5G RAN and verify that the RAN split functions are
deployed properly.

In [7]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True)))

NAME                            READY   STATUS    RESTARTS   AGE     IP              NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-t8jlb    1/1     Running   0          114s    10.244.134.6    node-core   <none>           <none>
open5gs-ausf-5778d7548b-4qsgm   1/1     Running   0          100s    10.244.134.7    node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-8rnt2    1/1     Running   0          15s     10.244.134.13   node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          2m34s   10.244.134.4    node-core   <none>           <none>
open5gs-nrf-65d98494d7-spl7d    1/1     Running   0          2m13s   10.244.134.5    node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-ll2jm   1/1     Running   0          29s     10.244.134.12   node-core   <none>           <none>
open5gs-pcf-6f444b66c5-hnjml    1/1     Running   0          73s     10.244.134.9    node-core   <none>           <none>
open5gs-smf-5c7c8d4c4c-

In [8]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)
await ran.deploy()


PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du2' chart] **********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-cu' chart] ***********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different inter

We can verify that all functions are deployed in the kubernetes cluster by
checking the pods deployed in the `oai-ran` namespace.

In [9]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov.deployment_model.params.k8s.master_node],
    become=True, quiet=True, flatten=False))[prov.deployment_model.params.k8s.master_node])

NAME                        READY   STATUS    RESTARTS   AGE   IP               NODE       NOMINATED NODE   READINESS GATES
oai-cu-5f99dffff4-4s578     1/1     Running   0          21s   10.244.129.168   node-ran   <none>           <none>
oai-cucp-59d9bb8f87-nbz87   1/1     Running   0          21s   10.244.41.38     node-ran   <none>           <none>
oai-cuup-75cffb6c9b-qz8jf   1/1     Running   0          11s   10.244.86.27     node-ran   <none>           <none>
oai-du1-84467fd8bd-mxphc    1/1     Running   0          2s    10.244.77.225    node-ran   <none>           <none>
oai-du2-7ff8477f44-rjl74    1/1     Running   0          2s    10.244.184.53    node-ran   <none>           <none>


## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the UEs we have
defined in our metamodel.

In [10]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]

TASK [helm/prepare : Copy custom Helm chart directory for 'oai-sim-ue2' if provided] ***
skipping: [node-core]

TASK [helm/prepare : Copy local built-in 'oai-containerized' Helm chart directory for 'oai-sim-ue2'] ***
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future ins

We can check that the two UEs have been deployed by listing the pods in the
`oai-ue` namespace.

In [11]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True)))

NAME                           READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue1-77fdccc764-dmspz   1/1     Running   0          10s   10.244.198.3   node-ues   <none>           <none>
oai-sim-ue2-6644849f47-ztv4b   1/1     Running   0          10s   10.244.198.2   node-ues   <none>           <none>


We can also analyse the logs the the UEs and see that they both successuflly
registered to the network and established a PDU session.

In [12]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

In [13]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [14]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True, flatten=False)

ping from `sim_ue1`:

In [15]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
[1792747484.930319] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=33.7 ms
[1792747485.938047] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=40.3 ms
[1792747486.944759] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=45.6 ms
[1792747487.940391] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=39.6 ms
[1792747488.947085] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=45.5 ms
[1792747489.939280] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=36.0 ms
[1792747490.942239] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=38.3 ms
[1792747491.944636] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=39.9 ms
[1792747492.954121] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=47.4 ms
[1792747493.950232] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=42.6 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9011ms
rtt min/avg/max/mdev = 33.694/40.888/47.405/4.179 ms



ping from `sim_ue2`:

In [16]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1792747484.886330] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=117 time=42.2 ms
[1792747485.870771] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=117 time=25.4 ms
[1792747486.891846] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=117 time=45.3 ms
[1792747487.885805] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=117 time=38.2 ms
[1792747488.889128] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=117 time=40.6 ms
[1792747489.893254] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=117 time=43.0 ms
[1792747490.891009] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=117 time=39.9 ms
[1792747491.890549] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=117 time=37.4 ms
[1792747492.898653] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=117 time=44.0 ms
[1792747493.891975] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=117 time=36.3 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9011ms
rtt min/avg/max/mdev = 25.422/39.218/45.279/5.362 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [17]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")